In [1]:
%load_ext cuml.accel
%run /workspace/alvin/SAR_ML/notebooks/SSR/SSRtransforms_preloaded.py
# %run /mnt/d/Users/Admin/Projects/dso/SAR_ML/notebooks/SSR/SSRtransforms_preloaded.py
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms, models
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, ConcatDataset
import SimpleITK as sitk
from scipy.ndimage import uniform_filter, binary_closing, binary_dilation, label, center_of_mass, distance_transform_edt
from skimage.morphology import remove_small_objects
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from cuml.manifold import TSNE, UMAP
from joblib import Parallel, delayed
from tqdm import tqdm

/opt/py_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def set_seed(seed=42):
    """Set all random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Uncomment only if you need 100% determinism and can handle errors
    # torch.use_deterministic_algorithms(True, warn_only=True)
    # os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    
    os.environ["PYTHONHASHSEED"] = str(seed)

def worker_init_fn(worker_id):
    """DataLoader worker init for reproducibility"""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

In [25]:
def anisotropic_diffusion(img, n_iter = 20, kappa = 0.1, gamma = 0.1):
    img = img if isinstance(img, np.ndarray) else img.numpy()
    sitk_img = sitk.GetImageFromArray(img.astype(np.float32))
    filt = sitk.GradientAnisotropicDiffusionImageFilter()
    filt.SetNumberOfIterations(n_iter)
    filt.SetConductanceParameter(kappa)
    filt.SetTimeStep(gamma)
    result = filt.Execute(sitk_img)
    return sitk.GetArrayFromImage(result)

def filter_shadow_by_distance(shadow_mask, T_mask, T_D=10, min_shadow_size=20):
    """
    Improved shadow filtering:
      1. remove_small_objects to eliminate tiny noise CCs
      2. Accept CC if its CLOSEST pixel to the target boundary is within T_D
         (boundary distance, not centroid-to-centroid distance)
    """
    # Step 1: remove small spurious CCs
    cleaned = remove_small_objects(shadow_mask.astype(bool), min_size=min_shadow_size)
    
    # Step 2: compute distance transform from target boundary
    # distance_transform_edt on the *inverted* target mask gives
    # each pixel its distance to the nearest target pixel
    dist_from_target = distance_transform_edt(1 - T_mask)
    
    labeled, n = label(cleaned)
    if n == 0:
        return np.zeros_like(shadow_mask)
    
    best_mask = np.zeros_like(shadow_mask)
    
    for region_id in range(1, n + 1):
        region = (labeled == region_id)
        # minimum distance from any pixel in this CC to the target boundary
        min_dist = dist_from_target[region].min()
        if min_dist < T_D:
            best_mask |= region
    
    return best_mask

def zhao_segmentation_v2(img):
    I_pm = anisotropic_diffusion(img)
    I_n = readjust_intensity(I_pm)
    T_b, S_b = find_target_and_shadow_mask(I_n, high = 90, low = 35)
    T_c, S_c = counting_filter(T_b, window_size = 5, threshold = 15), counting_filter(S_b, window_size = 5, threshold = 15)
    T_dilated, S_closing = binary_dilation(T_c, structure = structing_ele(shape = 5)).astype(int), binary_closing(S_c, structure = structing_ele(shape = 5)).astype(int)
    
    T_mask = find_largest_connected_component(T_dilated)
    if T_mask.sum() == 0:
        zeros = np.zeros(img.shape, dtype=int)
        return zeros, zeros, np.ones(img.shape, dtype=int)

    S_mask = filter_shadow_by_distance(S_closing, T_mask, T_D=5, min_shadow_size=20)

    clutter = 1 - (T_mask + S_mask)
    return T_mask, S_mask, clutter

# identity mapping: no augmentations
identity = lambda x: x

class MaskedAugmentation:
    """
    Computes Choi segmentation mask on the pre-augmentation image,
    applies augmentation, then masks the result to target+shadow only.
    
    Pipeline: log_mapped_img → compute mask → aug → apply mask
    
    Args:
        augmentation: SSRAugmentation, GaussianNoise, or any transform
                      that accepts (image, filepath) tuples
    """
    def __init__(self, augmentation, fill=np.mean, mask_cache=None):
        self.aug = augmentation
        self.fill = fill
        self.mask_cache = mask_cache

    def __call__(self, img_or_tuple):
        if isinstance(img_or_tuple, tuple):
            image, filepath = img_or_tuple
        else:
            # Can't segment without filepath context; passthrough
            return img_or_tuple
    
        # Use cached mask if available, else compute on the fly
        if self.mask_cache is not None and filepath in self.mask_cache:
            mask = self.mask_cache[filepath].astype(np.float32)
        else:
            M_target, M_shadow, _ = zhao_segmentation_v2(image)
            mask = (M_target + M_shadow).astype(np.float32)

        # Step 2: apply augmentation
        aug_result = self.aug((image, filepath))
        aug_image = aug_result[0] if isinstance(aug_result, tuple) else aug_result
        fp = aug_result[1] if isinstance(aug_result, tuple) else filepath
        
        # Step 3: apply mask + fill using augmented image statistics
        if mask.any():
            target_shadow_pixels = aug_image[mask == 1]
            fill_val = self.fill(target_shadow_pixels) if callable(self.fill) else float(self.fill)
        else:
            fill_val = 0.0

        result = aug_image * mask + fill_val * (1 - mask)

        return (result.astype(np.float32), fp)
    
class MaskedAugmentationSpeckleFill:
    """
    Computes Choi segmentation mask on the pre-augmentation image,
    applies augmentation, then masks the result to target+shadow only.
    
    Pipeline: log_mapped_img → compute mask → aug → apply mask
    
    Args:
        augmentation: SSRAugmentation, GaussianNoise, or any transform
                      that accepts (image, filepath) tuples
    """
    def __init__(self, augmentation, fill=np.mean, ray_sigma = 0.1, mask_cache=None):
        self.aug = augmentation
        self.fill = fill
        self.ray_sigma = ray_sigma
        self.mask_cache = mask_cache

    def __call__(self, img_or_tuple):
        if isinstance(img_or_tuple, tuple):
            image, filepath = img_or_tuple
        else:
            # Can't segment without filepath context; passthrough
            return img_or_tuple

        # Use cached mask if available, else compute on the fly
        if self.mask_cache is not None and filepath in self.mask_cache:
            mask = self.mask_cache[filepath].astype(np.float32)
        else:
            M_target, M_shadow, _ = zhao_segmentation_v2(image)
            mask = (M_target + M_shadow).astype(np.float32)

        # Step 2: apply augmentation
        aug_result = self.aug((image, filepath))
        aug_image = aug_result[0] if isinstance(aug_result, tuple) else aug_result
        fp = aug_result[1] if isinstance(aug_result, tuple) else filepath
        
        # Step 3: apply mask + fill using augmented image statistics
        if mask.any():
            target_shadow_pixels = aug_image[mask == 1]
            fill_val = self.fill(target_shadow_pixels) if callable(self.fill) else float(self.fill)
        else:
            fill_val = 0.0
        
        synthetic_clutter = np.random.rayleigh(self.ray_sigma, size=aug_image.shape).astype(np.float32)

        result = aug_image * mask + ((fill_val + synthetic_clutter)* (1 - mask))

        return (result.astype(np.float32), fp)

In [4]:
workspace = "/workspace/alvin/SAR_ML"
# workspace = "/mnt/d/Users/Admin/Projects/dso/SAR_ML"
data_workspace = os.path.join(workspace, "data/SAMPLE")

In [5]:
no_aug_synth_ds = synth_ds = DatasetFolderWithPath(
    os.path.join(data_workspace, "mat_files/synth"),
    extensions = (".mat"),
    transform = transforms.Compose(
        [Magnitude(), 
         LogMapping(c = 1000.0), 
         NumpyToTensor3Channel()]
    ),
    loader = mat_file_loader
)

meas_ds = datasets.DatasetFolder(
    os.path.join(data_workspace, "mat_files/real"),
    extensions = (".mat"),
    transform = transforms.Compose(
        [Magnitude(), 
         LogMapping(c = 1000.0), 
         NumpyToTensor3Channel()]),
    loader = mat_file_loader
)

In [6]:
synth_paths = [s[0] for s in no_aug_synth_ds.samples]

synth_mask_cache = {}
for path in tqdm(synth_paths):
    img = mat_file_loader(path)
    img = LogMapping(c=1000.0)(Magnitude()(img))
    T, S, _ = zhao_segmentation_v2(img)
    synth_mask_cache[path] = (T + S).astype(np.uint8)

meas_paths = [s[0] for s in meas_ds.samples]

meas_mask_cache = {}
for path in tqdm(meas_paths):
    img = mat_file_loader(path)
    img = LogMapping(c=1000.0)(Magnitude()(img))
    T, S, _ = zhao_segmentation_v2(img)
    meas_mask_cache[path] = (T + S).astype(np.uint8)

100%|██████████| 1345/1345 [03:20<00:00,  6.71it/s]


In [28]:
seg_synth_ds = DatasetFolderWithPath(
    os.path.join(data_workspace, "mat_files/synth"),
    extensions=(".mat"),
    transform=transforms.Compose([
        Magnitude(),
        LogMapping(c=1000.0),
        MaskedAugmentation(augmentation=identity, fill=0, mask_cache=synth_mask_cache),
        NumpyToTensor3Channel()
    ]),
    loader=mat_file_loader
)

seg_meas_ds = DatasetFolderWithPath(
    os.path.join(data_workspace, "mat_files/real"),
    extensions=(".mat"),
    transform=transforms.Compose([
        Magnitude(),
        LogMapping(c=1000.0),
        MaskedAugmentation(augmentation=identity, fill=0, mask_cache=meas_mask_cache),
        NumpyToTensor3Channel()
    ]),
    loader=mat_file_loader
)

seg_mean_synth_ds = DatasetFolderWithPath(
    os.path.join(data_workspace, "mat_files/synth"),
    extensions=(".mat"),
    transform=transforms.Compose([
        Magnitude(),
        LogMapping(c=1000.0),
        MaskedAugmentation(augmentation=identity, fill=np.mean, mask_cache=synth_mask_cache),
        NumpyToTensor3Channel()
    ]),
    loader=mat_file_loader
)

seg_mean_meas_ds = DatasetFolderWithPath(
    os.path.join(data_workspace, "mat_files/real"),
    extensions=(".mat"),
    transform=transforms.Compose([
        Magnitude(),
        LogMapping(c=1000.0),
        MaskedAugmentation(augmentation=identity, fill=np.mean, mask_cache=meas_mask_cache),
        NumpyToTensor3Channel()
    ]),
    loader=mat_file_loader
)

seg_speckle_synth_ds = DatasetFolderWithPath(
    os.path.join(data_workspace, "mat_files/synth"),
    extensions=(".mat"),
    transform=transforms.Compose([
        Magnitude(),
        LogMapping(c=1000.0),
        MaskedAugmentationSpeckleFill(augmentation=identity, fill=np.mean, ray_sigma=0.1, mask_cache=synth_mask_cache),
        NumpyToTensor3Channel()
    ]),
    loader=mat_file_loader
)

seg_speckle_meas_ds = DatasetFolderWithPath(
    os.path.join(data_workspace, "mat_files/real"),
    extensions=(".mat"),
    transform=transforms.Compose([
        Magnitude(),
        LogMapping(c=1000.0),
        MaskedAugmentationSpeckleFill(augmentation=identity, fill=np.mean, ray_sigma=0.1, mask_cache=meas_mask_cache),
        NumpyToTensor3Channel()
    ]),
    loader=mat_file_loader
)

In [29]:
train_ds = seg_speckle_synth_ds
test_ds = seg_speckle_meas_ds

ds_dict = {"train" : train_ds, "test": test_ds}
dataset_sizes = {"train" : len(train_ds), "test": len(test_ds)}

In [1]:
seed_lst = [10, 42, 100, 123, 666, 777, 849, 1000, 1111, 1234]

train_loss = []
# val_loss = []
train_acc =[]
# val_acc = []

for i, seed in enumerate(seed_lst):
    print(f"Training Run {i}: seed {seed}")

    set_seed(seed)

    dataloaders = {
        "train": DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=12, pin_memory = True, persistent_workers=True, worker_init_fn=worker_init_fn, generator=torch.Generator().manual_seed(seed)),
        # "val": DataLoader(valid_ds, batch_size=16, shuffle=False, num_workers=8, pin_memory = True, persistent_workers=True, worker_init_fn=worker_init_fn, generator=torch.Generator().manual_seed(seed)),
        "test": DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=12, pin_memory = True, persistent_workers=True, worker_init_fn=worker_init_fn, generator=torch.Generator().manual_seed(seed))
    }
    # load pre-trained model
    model = models.resnet18(weights = None)

    # Replace final layer for the number of classes
    model.fc = nn.Sequential(
        nn.Dropout(p = 0.4),
        nn.Linear(model.fc.in_features, len(train_ds.class_to_idx))
    )
    
    # Define the loss function and optimizer
    criterion = nn.CrossEntropyLoss() # most common used nn for classification problems
    
    optimizer = optim.AdamW(model.parameters(), lr = 3e-4, weight_decay = 2e-4)
    
    scheduler = CosineAnnealingLR(optimizer, T_max = 200, eta_min = 3e-7)
    # scheduler = OneCycleLR(
    #     optimizer, 
    #     max_lr = 2.5e-4, 
    #     steps_per_epoch = len(dataloaders["train"]), 
    #     epochs = 100
    #     )
        
    # move model to GPU
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    
    history = {
        "train_loss": [],
        # "val_loss" : [],
        "train_acc": [],
        # "val_acc" : []
    }
    
    # Training loops
    num_epochs = 200
    for epoch in range(num_epochs):
        print(f"Epoch {epoch}")
        if epoch == 0:
            print(f"First layer mean: {model.conv1.weight.data.mean():.6f}")
        for phase in ["train"]:
            if phase == "train":
                model.train()
            else:
                model.eval()
    
            running_loss = 0.0
            running_corrects = 0 # correct predictions
    
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)
    
                optimizer.zero_grad() # clear the gradient from previous iteration
    
                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels) # check if output and labels match
    
                    if phase == "train":
                        loss.backward()
                        optimizer.step()
                        # scheduler.step() # scheduler here if OneCycleLR
    
                running_loss += loss.item() * inputs.size(0)
                running_corrects += (preds == labels).sum().item()
    
            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects / dataset_sizes[phase]
            
            history[f"{phase}_loss"].append(epoch_loss)
            history[f"{phase}_acc"].append(epoch_acc)
    
            print(f"{phase} Loss: {epoch_loss:.10f} Acc: {epoch_acc:.10f}")
    
        scheduler.step()
        print(f"Epoch {epoch} LR: {scheduler.get_last_lr()[0]:.10f}")
        
    print("Training complete!")
    
    train_loss.append(history["train_loss"])
    # val_loss.append(history["val_loss"])
    train_acc.append(history["train_acc"])
    # val_acc.append(history["val_acc"])
    
    torch.save(model.state_dict(), os.path.join(workspace, f"weights/SSR/Segmentation/Zhao_v2/wo_aug/rn18_seed{seed}_b16_speckle .pth"))

train_loss = np.array(train_loss)
# val_loss = np.array(val_loss)
train_acc = np.array(train_acc)
# val_acc = np.array(val_acc)

Training Run 0: seed 10


NameError: name 'set_seed' is not defined

# Results

## Fill = 0

In [12]:
seed_lst = [10, 42, 100, 123, 666, 777, 849, 1000, 1111, 1234]

# Evaluate all 10 trained models on test set
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
test_acc_lst = []
for i, seed in enumerate(seed_lst):
    print(f"Evaluating Run {i}: Seed {seed}")
    new_model = models.resnet18(weights = None) # dont load ImageNet Weights
    new_model.fc = nn.Sequential(
        nn.Dropout(p = 0.4),
        nn.Linear(new_model.fc.in_features, len(train_ds.class_to_idx))
    )
    
    # Load your trained weights
    new_model.load_state_dict(torch.load(
        os.path.join(workspace, f"weights/SSR/Segmentation/Zhao_v2/wo_aug/rn18_seed{seed}_b16.pth"),
        map_location=device
    ))
    
    new_model = new_model.to(device)
    new_model.eval()

    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in DataLoader(seg_meas_ds, batch_size=16, shuffle=False, num_workers=8, pin_memory = True, persistent_workers=True):
            inputs = inputs.to(device)
            labels = labels.to(device)
    
            outputs = new_model(inputs)
            _, preds = torch.max(outputs, 1)
    
            correct += torch.sum(preds == labels).item()
            total += labels.size(0)
    
    test_acc = correct / total
    print(f"Test Accuracy: {test_acc:.4f}")
    test_acc_lst.append(test_acc)

Evaluating Run 0: Seed 10
Test Accuracy: 0.8126
Evaluating Run 1: Seed 42
Test Accuracy: 0.8283
Evaluating Run 2: Seed 100
Test Accuracy: 0.7911
Evaluating Run 3: Seed 123
Test Accuracy: 0.8245
Evaluating Run 4: Seed 666
Test Accuracy: 0.8000
Evaluating Run 5: Seed 777
Test Accuracy: 0.8082
Evaluating Run 6: Seed 849
Test Accuracy: 0.7933
Evaluating Run 7: Seed 1000
Test Accuracy: 0.8335
Evaluating Run 8: Seed 1111
Test Accuracy: 0.7896
Evaluating Run 9: Seed 1234
Test Accuracy: 0.8491


In [17]:
test_acc_arr = np.array(test_acc_lst)
print(f"Min: {test_acc_arr.min() * 100:.4f}")
print(f"Max: {test_acc_arr.max() * 100:.4f}")
print(f"Avg, Std: {test_acc_arr.mean() * 100:.4f}, {test_acc_arr.std() * 100:.4f}")

Min: 78.9591
Max: 84.9071
Avg, Std: 81.3011, 1.9220


## Mean Filled

In [23]:
seed_lst = [10, 42, 100, 123, 666, 777, 849, 1000, 1111, 1234]

# Evaluate all 10 trained models on test set
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
test_acc_lst = []
for i, seed in enumerate(seed_lst):
    print(f"Evaluating Run {i}: Seed {seed}")
    new_model = models.resnet18(weights = None) # dont load ImageNet Weights
    new_model.fc = nn.Sequential(
        nn.Dropout(p = 0.4),
        nn.Linear(new_model.fc.in_features, len(train_ds.class_to_idx))
    )
    
    # Load your trained weights
    new_model.load_state_dict(torch.load(
        os.path.join(workspace, f"weights/SSR/Segmentation/Zhao_v2/wo_aug/rn18_seed{seed}_b16_mean_filled.pth"),
        map_location=device
    ))
    
    new_model = new_model.to(device)
    new_model.eval()

    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in DataLoader(seg_mean_meas_ds, batch_size=16, shuffle=False, num_workers=8, pin_memory = True, persistent_workers=True):
            inputs = inputs.to(device)
            labels = labels.to(device)
    
            outputs = new_model(inputs)
            _, preds = torch.max(outputs, 1)
    
            correct += torch.sum(preds == labels).item()
            total += labels.size(0)
    
    test_acc = correct / total
    print(f"Test Accuracy: {test_acc:.4f}")
    test_acc_lst.append(test_acc)

Evaluating Run 0: Seed 10
Test Accuracy: 0.8892
Evaluating Run 1: Seed 42
Test Accuracy: 0.8349
Evaluating Run 2: Seed 100
Test Accuracy: 0.8089
Evaluating Run 3: Seed 123
Test Accuracy: 0.8401
Evaluating Run 4: Seed 666
Test Accuracy: 0.8149
Evaluating Run 5: Seed 777
Test Accuracy: 0.8691
Evaluating Run 6: Seed 849
Test Accuracy: 0.7926
Evaluating Run 7: Seed 1000
Test Accuracy: 0.7777
Evaluating Run 8: Seed 1111
Test Accuracy: 0.8461
Evaluating Run 9: Seed 1234
Test Accuracy: 0.9011


In [24]:
test_acc_arr = np.array(test_acc_lst)
print(f"Min: {test_acc_arr.min() * 100:.4f}")
print(f"Max: {test_acc_arr.max() * 100:.4f}")
print(f"Avg, Std: {test_acc_arr.mean() * 100:.4f}, {test_acc_arr.std() * 100:.4f}")

Min: 77.7695
Max: 90.1115
Avg, Std: 83.7472, 3.8386
